In [ ]:
import json
import sqlite3
import pandas as pd
from dotenv import load_dotenv
import aisuite as ai

load_dotenv()
client = ai.Client()

In [ ]:
DB_PATH = "library_database.db"

def get_schema(db_path: str) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT name, sql FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%';
    """)
    rows = cur.fetchall()
    conn.close()
    return "\n\n".join(f"Table: {name}\n{sql}" for name, sql in rows)

schema_block = get_schema(DB_PATH)
print(schema_block)

In [ ]:
def execute_sql(query: str) -> pd.DataFrame:
    """Execute a SQLite query and return results as a DataFrame."""
    q = query.strip().removeprefix("```sql").removesuffix("```").strip()
    conn = sqlite3.connect(DB_PATH)
    try:
        return pd.read_sql_query(q, conn)
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})
    finally:
        conn.close()

TOOL_MAP = {"execute_sql": execute_sql}

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_sql",
            "description": (
                "Execute a SQLite SELECT query against the library database "
                "and return the results. Always use this tool to fetch data — "
                "never invent or assume values."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A complete, valid SQLite SELECT statement.",
                    }
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    }
]

In [90]:
def run_agent(query: str, max_turns: int = 5) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a friendly, conversational library assistant. "
                "Chat naturally — never mention SQL, databases, tables, or technical terms in your replies.\n\n"
                f"Database schema:\n{schema_block}\n\n"
                "Critical rules for querying:\n"
                "- members.active_checkouts = CURRENT open books only (a snapshot counter). "
                "NEVER use it to rank or count total borrowing history.\n"
                "- To count ALL books ever borrowed by a member, use: "
                "SELECT m.name, COUNT(*) as total FROM transactions t "
                "JOIN members m ON t.member_id = m.member_id "
                "WHERE t.txn_type = 'checkout' GROUP BY t.member_id ORDER BY total DESC.\n"
                "- To find overdue books: transactions WHERE txn_type = 'checkout' "
                "AND status = 'open' AND due_date < date('now').\n"
                "- Always JOIN members / catalog when you need names or titles.\n"
                "- Always use the execute_sql tool to retrieve data. Never invent values."
            ),
        },
        {"role": "user", "content": query},
    ]

    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model="openai:gpt-4o",
            temperature=0.0,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        choice = resp.choices[0]
        msg = choice.message
        messages.append(msg.model_dump(exclude_none=True))

        if choice.finish_reason == "tool_calls":
            for tc in msg.tool_calls:
                fn_args = json.loads(tc.function.arguments)
                result_df = TOOL_MAP[tc.function.name](**fn_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_df.to_string(index=False),
                })

        elif choice.finish_reason == "stop":
            return msg.content

    return "I wasn't able to find an answer. Could you rephrase your question?"

In [91]:
print("Library Assistant ready. Type 'exit' to quit.\n")
while True:
    user_input = input("You: ").strip()
    if not user_input or user_input.lower() == "exit":
        print("Goodbye!")
        break
    reply = run_agent(user_input)
    print(f"\nAssistant: {reply}\n")

Library Assistant ready. Type 'exit' to quit.


Assistant: Hello! How can I assist you today? Are you looking for a book, need help with your membership, or something else?


Assistant: Here are some books that are currently available:

1. **Dune** by Frank Herbert
2. **1984** by George Orwell
3. **Sapiens** by Yuval Noah Harari
4. **The Pragmatic Programmer** by Hunt & Thomas
5. **The Great Gatsby** by F. Scott Fitzgerald
6. **Clean Code** by Robert C. Martin
7. **The Alchemist** by Paulo Coelho
8. **Thinking, Fast and Slow** by Daniel Kahneman
9. **The Hobbit** by J.R.R. Tolkien
10. **Brave New World** by Aldous Huxley
11. **Deep Work** by Cal Newport
12. **The Lean Startup** by Eric Ries
13. **Foundation** by Isaac Asimov
14. **Neuromancer** by William Gibson
15. **Harry Potter and the Sorcerer's Stone** by J.K. Rowling
16. **To Kill a Mockingbird** by Harper Lee
17. **The Power of Habit** by Charles Duhigg
18. **Python Crash Course** by Eric Matthes
19. **The Martian** by Andy Weir